# The napari plugin

Four widgets, all in one napari window:

| widget | for |
|---|---|
| 1. Training regions | see the test regions of a brain, draw new ones |
| 2. Annotate and train | annotate candidates, run networks, statistics, train |
| 3. Whole-brain results | cells and cell density of a whole-brain run |
| 4. Atlas regions | register to the atlas, cells per region |

The usual way to start it is a terminal on a machine with a screen (desktop session on ba-training):

```bash
cfos-mapper napari --settings /path/to/settings.json
```

It also works from a notebook (with a screen), as below; the widgets can then be driven from code.

In [ ]:
%gui qt

import os

# the settings file of the project (where the brains are, where results go, all parameters)
os.environ["CFOS_MAPPER_SETTINGS"] = "/mif-users/no_backup/Gritti_Nicola/IDIBAPS-brains/code/settings.json"

from cfos_mapper.plugin import open_viewer

viewer = open_viewer()   # napari window with the four widgets as tabs

## Driving the widgets from code

Or make the widgets yourself and keep a handle on them:

In [ ]:
import napari
from cfos_mapper.plugin import AnnotationWidget, AtlasWidget, RegionsWidget, ResultsWidget

viewer = napari.Viewer(title="cFos mapper")
regions = RegionsWidget(viewer)
annotate = AnnotationWidget(viewer)
results = ResultsWidget(viewer)
atlas = AtlasWidget(viewer)
for widget, name in [(regions, "1. Training regions"), (annotate, "2. Annotate and train"),
                     (results, "3. Whole-brain results"), (atlas, "4. Atlas regions")]:
    viewer.window.add_dock_widget(widget, name=name, area="right", tabify=True)

## 1. Training regions

*Load brain* shows the whole brain (all resolution levels, loaded only where you look) with the saved test
regions outlined and named. *Draw a new region*: drag a rectangle; the third axis takes the size in the boxes
(or roll the view and draw a second rectangle). Check the yellow preview, name it, *Save region*: candidates are
found in the background.

In [ ]:
regions.brain_box.setCurrentText("O16")
regions.load_brain()

In [ ]:
from napari.utils import nbscreenshot
nbscreenshot(viewer)

## 2. Annotate and train

1. choose a region, *Load region*: candidates appear as white rings
2. select candidates (candidates layer, arrow tool) and press **c** (cell), **x** (not a cell), **d** (delete)
3. choose a network, *Run network on this region*: light green / red rings, magenta = it disagrees with you;
   **n** / **Shift+N** go through the disagreements
4. *Save annotations* (often); *Train new network* when you have a few hundred more

*How to use* opens the full instructions next to napari.

In [ ]:
annotate.region_box.setCurrentText("O16 / regionA_cortex")
annotate.load_region()          # runs in the background; the layers appear when it is loaded

In [ ]:
annotate.network_box.setCurrentText("pretrained")
annotate.run_network()          # first time: a background job on the GPU (~1 min)
print(annotate.statistics.text())

## 3. Whole-brain results

Choose a brain and a run (the network used for the whole brain), *Load*: the brain, the cells (light green
rings, not cells in light red if ticked) and the cell density (blue).

In [ ]:
results.brain_box.setCurrentText("O15")
results.load()

## 4. Atlas regions

*Register this brain* (once, ~30-60 min in the background), then *Show atlas on the brain*: the atlas outlines
should follow the brain; hover to read the region name in the status bar. *Count cells per region* for a
whole-brain run, then *Show density per region*: every region coloured by its cells per mm³.

In [ ]:
atlas.brain_box.setCurrentText("O15")
atlas.show_atlas()